In [ ]:
#配置环境
!pip install --no-cache-dir \
  ms-swift==3.5.0 \
  transformers==4.51.3 \
  datasets==3.2.0 \
  peft==0.15.2 \
  trl==0.17.0

In [ ]:
pip uninstall -y apex

In [ ]:
#微调前调用
import os
from swift.llm import ModelType, InferArguments, infer_main

os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# 系统提示
SYSTEM_PROMPT = (
    "Answer the following questions as best you can. You have access to the following APIs:\n"
    "1. trailFinder: Call this tool to interact with the trailFinder API. What is the trailFinder API useful for? "
    "API for finding nearby hiking trails based on user input. Parameters: [{\"name\": \"location\", \"description\": "
    "\"User's current location.\", \"required\": \"True\"}, {\"name\": \"distance\", \"description\": \"Maximum "
    "distance from user's location.\", \"required\": \"False\"}, {\"name\": \"difficulty\", \"description\": "
    "\"Specify the difficulty level of the trail.\", \"required\": \"False\"}]\n\n"
    "2. Factorial calculator: Call this tool to interact with the Factorial calculator API. What is the Factorial "
    "calculator API useful for? 计算正整数的阶乘. Parameters: [{\"name\": \"n\", \"description\": "
    "\"需要计算阶乘的正整数\", \"required\": \"False\"}]\n\n"
    "3. weather: Call this tool to interact with the weather API. What is the weather API useful for? "
    "天气查询API，查询指定城市的实时天气情况. Parameters: [{\"name\": \"city\", \"description\": "
    "\"指定查询的城市名称\", \"required\": \"False\"}, {\"name\": \"date\", \"description\": "
    "\"指定查询的日期\", \"required\": \"False\"}]\n\n"
    "4. English to Chinese Translator: Call this tool to interact with the English to Chinese Translator API. What "
    "is the English to Chinese Translator API useful for? 将英文翻译成中文. Parameters: [{\"name\": \"english_text\", "
    "\"description\": \"需要翻译的英文文本\", \"required\": \"True\"}, {\"name\": \"target_language\", "
    "\"description\": \"目标语言（默认为中文）\", \"required\": \"False\"}]\n\n"
    "Use the following format:\n\n"
    "Thought: you should always think about what to do\n"
    "Action: the action to take, should be one of the above tools[trailFinder, Factorial calculator, weather, "
    "English to Chinese Translator]\n"
    "Action Input: the input to the action\n"
    "Observation: the result of the action\n"
    "... (this Thought/Action/Action Input/Observation can be repeated indirect_weatherero or more times)\n"
    "Thought: I now know the final answer\n"
    "Final Answer: the final answer to the original input question\n"
    "Begin!"
)

# 使用 Qwen1.5-0.5B-Chat 模型进行推理，并加载指定的 checkpoint
infer_args = InferArguments(

    model="Qwen/Qwen1.5-0.5B-Chat",
    max_model_len=2048,
    system=SYSTEM_PROMPT  # 将系统提示赋值给 system 属性
)

infer_main(infer_args)

In [ ]:
#下载数据集 
%cd /mnt/workspace/Advanced_AI_Course/Lab_Lora
!git clone https://www.modelscope.cn/datasets/iic/ms_agent.git

In [ ]:
#处理数据集
import os
import jsonlines

# File paths
input_file = "/mnt/workspace/Advanced_AI_Course/Lab_Lora/ms_agent/train_agent_react.jsonl"
output_file = "/mnt/workspace/Advanced_AI_Course/Lab_Lora/train-weather.jsonl"


# Check if input file exists
if not os.path.exists(input_file):
    print(f"File not found: {input_file}")
else:
    # First, filter records containing "天气" in user messages
    with jsonlines.open(input_file, 'r') as reader, jsonlines.open(output_file, 'w') as writer:
        for obj in reader:
            keep_record = False
            if "conversations" in obj:
                for message in obj["conversations"]:
                    if message.get("from") == "user" and "天气" in message.get("value", ""):
                        keep_record = True
                        break
            if keep_record:
                writer.write(obj)

    # Then, filter lines with fewer than 2048 characters
    temp_file = output_file + ".tmp"
    with open(output_file, 'r', encoding='utf-8') as infile, open(temp_file, 'w', encoding='utf-8') as outfile:
        for line in infile:
            char_count = len(line.strip())
            if char_count < 2048:
                outfile.write(line)

    # Replace the original output file with the filtered one
    os.replace(temp_file, output_file)

    print(f"Processing complete. Results saved to {output_file}")

In [ ]:
#lora微调 checkpoint
!CUDA_VISIBLE_DEVICES=0 \
swift sft \
    --model Qwen/Qwen1.5-0.5B-Chat \
    --train_type lora \
    --dataset=/mnt/workspace/Advanced_AI_Course/Lab_Lora/train-weather.jsonl \
    --torch_dtype bfloat16 \
    --output_dir output/lora/weather \
    --num_train_epochs=8 \
    --max_length=2048 \
    --lora_rank=8 \
    --lora_alpha=32 \
    --lora_dropout=0.05 \
    --target_modules all-linear \
    --model_name="qwen1.5-0.5B Chat" \
    --model_author="gkd" \
    --gradient_checkpointing=true \
    --per_device_train_batch_size 1 \
    --weight_decay=0.1 \
    --learning_rate=5e-5 \
    --gradient_accumulation_steps=8 \
    --max_grad_norm=0.5 \
    --warmup_ratio=0.03 \
    --eval_steps=100 \
    --save_steps=100 \
    --save_total_limit=2 \
    --logging_steps=10 \
    --dataloader_num_workers 0

In [ ]:
#微调后调用
import os
from dataclasses import field
from swift.llm import ModelType, InferArguments, infer_main

os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# 系统提示
SYSTEM_PROMPT = (
    "Answer the following questions as best you can. You have access to the following APIs:\n"
    "1. trailFinder: Call this tool to interact with the trailFinder API. What is the trailFinder API useful for? "
    "API for finding nearby hiking trails based on user input. Parameters: [{\"name\": \"location\", \"description\": "
    "\"User's current location.\", \"required\": \"True\"}, {\"name\": \"distance\", \"description\": \"Maximum "
    "distance from user's location.\", \"required\": \"False\"}, {\"name\": \"difficulty\", \"description\": "
    "\"Specify the difficulty level of the trail.\", \"required\": \"False\"}]\n\n"
    "2. Factorial calculator: Call this tool to interact with the Factorial calculator API. What is the Factorial "
    "calculator API useful for? 计算正整数的阶乘. Parameters: [{\"name\": \"n\", \"description\": "
    "\"需要计算阶乘的正整数\", \"required\": \"False\"}]\n\n"
    "3. weather: Call this tool to interact with the weather API. What is the weather API useful for? "
    "天气查询API，查询指定城市的实时天气情况. Parameters: [{\"name\": \"city\", \"description\": "
    "\"指定查询的城市名称\", \"required\": \"False\"}, {\"name\": \"date\", \"description\": "
    "\"指定查询的日期\", \"required\": \"False\"}]\n\n"
    "4. English to Chinese Translator: Call this tool to interact with the English to Chinese Translator API. What "
    "is the English to Chinese Translator API useful for? 将英文翻译成中文. Parameters: [{\"name\": \"english_text\", "
    "\"description\": \"需要翻译的英文文本\", \"required\": \"True\"}, {\"name\": \"target_language\", "
    "\"description\": \"目标语言（默认为中文）\", \"required\": \"False\"}]\n\n"
    "Use the following format:\n\n"
    "Thought: you should always think about what to do\n"
    "Action: the action to take, should be one of the above tools[trailFinder, Factorial calculator, weather, "
    "English to Chinese Translator]\n"
    "Action Input: the input to the action\n"
    "Observation: the result of the action\n"
    "... (this Thought/Action/Action Input/Observation can be repeated indirect_weatherero or more times)\n"
    "Thought: I now know the final answer\n"
    "Final Answer: the final answer to the original input question\n"
    "Begin!"
)

# 使用 Qwen1.5-0.5B 模型进行推理，并加载指定的 checkpoint
# 微调后的checkpoint保存在当前文件夹的output文件中 类似/mnt/workspace/Advanced_AI_Course/Lab_Lora/output/lora/weather/v0-20260408-210929/checkpoint-1000
checkpoint_path = "换成自己的checkpoint路径,要以/mnt/workspace/开头的绝对路径或者相对路径"

# 使用 Qwen1.5-0.5B-Chat 模型进行推理，并加载指定的 checkpoint
infer_args = InferArguments(

    model="Qwen/Qwen1.5-0.5B-Chat",
    max_model_len=2048,
    ckpt_dir=checkpoint_path,
    system=SYSTEM_PROMPT  # 将系统提示赋值给 system 属性
)

infer_main(infer_args)